# Train trên Kaggle (GPU miễn phí)

Hướng dẫn nhanh:
1. Tạo notebook mới trên Kaggle: https://www.kaggle.com/code → **New Notebook**.
2. **Settings → Accelerator → GPU T4 x2** (hoặc P100).
3. **Add Input → Datasets**, tìm `31 classes of skin disease` (tác giả kelixo25) và thêm vào.
4. Mở **File → Upload Notebook** rồi tải file này lên, hoặc copy từng cell.
5. Chạy lần lượt các cell bên dưới.

> Lưu ý: thay `GITHUB_REPO_URL` bằng URL repo của bạn sau khi đã push code lên GitHub.

In [ ]:
# 1) Kiểm tra GPU
import torch
print('CUDA:', torch.cuda.is_available())
print('GPU :', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')

In [ ]:
# 2) Lấy code. Cách A: clone từ GitHub (khuyến nghị)
GITHUB_REPO_URL = 'https://github.com/<user>/<repo>.git'  # <-- THAY URL repo của bạn
!git clone $GITHUB_REPO_URL project
%cd project

In [ ]:
# 3) Tìm đường dẫn dataset đã add vào Kaggle (thường ở /kaggle/input/...)
import os, glob
for p in glob.glob('/kaggle/input/*'):
    print(p)
    for sub in glob.glob(p + '/*')[:5]:
        print('   ', sub)

In [ ]:
# 4) Trỏ biến môi trường tới dataset Kaggle và nơi lưu output.
#    DATA_DIR phải là thư mục chứa 2 thư mục con 'train' và 'test'.
#    Sửa lại cho khớp output ở cell trên.
import os
os.environ['SKIN_DATA_ROOT'] = '/kaggle/input/31-classes-of-skin-disease/...'  # <-- THAY cho khớp
os.environ['SKIN_OUTPUT_ROOT'] = '/kaggle/working/outputs'

# Kiểm tra nhanh
import glob
print('train:', len(glob.glob(os.environ['SKIN_DATA_ROOT'] + '/train/*')), 'lớp')
print('test :', len(glob.glob(os.environ['SKIN_DATA_ROOT'] + '/test/*')), 'lớp')

In [ ]:
# 5) Cài thư viện còn thiếu (Kaggle đã có sẵn torch GPU, pandas, sklearn...)
!pip install -q imagehash grad-cam

In [ ]:
# 6) Chuẩn bị dữ liệu: khảo sát, kiểm tra ảnh lỗi, phát hiện trùng, tạo split
!python -m src.data.explore_dataset
!python -m src.data.validate_images
!python -m src.data.detect_duplicates
!python -m src.data.create_validation_split

In [ ]:
# 7) Train toàn bộ 4 mô hình trên GPU (nhanh hơn CPU nhiều lần)
!python -m src.training.train --all-models

In [ ]:
# 8) Đánh giá + sinh Grad-CAM + báo cáo tổng hợp
!python -m src.evaluation.evaluate --all-models
!python -m src.explainability.generate_gradcam_samples --all-models
!python -m src.evaluation.generate_summary

In [ ]:
# 9) Xem bảng so sánh và đóng gói kết quả để tải về
import pandas as pd
print(pd.read_csv('/kaggle/working/outputs/reports/model_comparison.csv'))

# Nén toàn bộ outputs để tải về (nút Output bên phải của Kaggle).
!cd /kaggle/working && zip -r outputs.zip outputs > /dev/null && echo 'Đã tạo /kaggle/working/outputs.zip'